# 04 — Standalone Memory Footprint: Compressed SGT-QAT Checkpoint

Notebook 03's benchmark couldn't measure the *genuinely compressed* SGT-QAT
checkpoint's memory footprint, because vLLM's `method="draft_model"` path can't
load compressed-tensors packed checkpoints at all (see `docs/findings.md`
2026-07-24) — we had to decompress to a plain ~3.4GB checkpoint just to get it
loading, which defeats the point of measuring compression's memory benefit.

This notebook answers a narrower, still-useful question directly: **what is the
compressed checkpoint's real VRAM footprint when loaded on its own** (no vLLM, no
target model, no speculative decoding — just load the model and read GPU memory)?
This is real, measured data, not an estimate — but it's not a perfect apples-to-
apples comparison against EAGLE-3's notebook-02 number, since that was measured
*inside* actual vLLM serving (target model + drafter + KV cache together), while
this is a standalone weight-only load. State that caveat explicitly wherever these
numbers get used.

**Not yet run.**

## Setup

In [ ]:
import os
REPO_NAME = 'sgt-qat-draft'
if not os.path.isdir(REPO_NAME):
    !git clone https://github.com/Resh19S/sgt-qat-draft.git
%cd {REPO_NAME}
!git pull

!pip install -q compressed-tensors

import torch, json, subprocess
from pathlib import Path
from datetime import datetime, timezone

assert torch.cuda.is_available(), "No GPU detected."
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

## Load the compressed checkpoint from Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CHECKPOINT = Path('/content/drive/MyDrive/sgt-qat-draft-checkpoints/qwen3-1.7b-sgt-qat')
LOCAL_CHECKPOINT = Path('checkpoints/qwen3-1.7b-sgt-qat')

if not LOCAL_CHECKPOINT.exists():
    LOCAL_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    !cp -r {str(DRIVE_CHECKPOINT)} {str(LOCAL_CHECKPOINT)}

!du -sh {str(LOCAL_CHECKPOINT)}  # sanity check: should be ~1.2GB (compressed), matching notebook 01

## Measure VRAM footprint (compressed, NOT decompressed)

Deliberately does NOT call `ModelCompressor.decompress_model()` here (unlike
notebook 03) -- the whole point is to measure the checkpoint in its genuinely
compressed, packed-in-memory form, which is what `AutoModelForCausalLM.from_pretrained()`
gives you by default (confirmed in notebook 03's debugging: it keeps weights in
`CompressedLinear` modules, packed, dequantizing only on-the-fly at inference).

In [ ]:
def _gpu_memory_used_mb(device_index: int = 0) -> int:
    out = subprocess.check_output([
        'nvidia-smi', f'--id={device_index}',
        '--query-gpu=memory.used', '--format=csv,noheader,nounits',
    ])
    return int(out.decode().strip().splitlines()[0]) * 1024 * 1024


baseline_memory_bytes = _gpu_memory_used_mb()
print(f"GPU memory before loading anything: {baseline_memory_bytes / 1024**3:.2f} GiB")

from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    str(LOCAL_CHECKPOINT), dtype=torch.float16, trust_remote_code=True, device_map='cuda',
)

compressed_memory_bytes = _gpu_memory_used_mb()
compressed_delta_bytes = compressed_memory_bytes - baseline_memory_bytes
print(f"GPU memory after loading compressed checkpoint: {compressed_memory_bytes / 1024**3:.2f} GiB")
print(f"Delta (checkpoint's own footprint): {compressed_delta_bytes / 1024**3:.3f} GiB")

## For comparison: measure the plain (decompressed) checkpoint too

Same idea, but for the plain checkpoint notebook 03 actually benchmarked through
vLLM -- gives a same-methodology (standalone load, not inside vLLM) comparison
point between compressed and plain, isolating the compression effect from the
vLLM-serving-overhead confound.

In [ ]:
PLAIN_CHECKPOINT = Path('/content/drive/MyDrive/sgt-qat-draft-checkpoints/qwen3-1.7b-sgt-qat-plain')
LOCAL_PLAIN_CHECKPOINT = Path('checkpoints/qwen3-1.7b-sgt-qat-plain')

# The plain checkpoint currently only exists wherever notebook 03 last produced
# it (local Colab disk, not necessarily backed up to Drive yet). If it's not on
# Drive, this cell will need notebook 03's decompression cell re-run here instead
# -- check before assuming this works unmodified.
if not LOCAL_PLAIN_CHECKPOINT.exists():
    if PLAIN_CHECKPOINT.exists():
        LOCAL_PLAIN_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
        !cp -r {str(PLAIN_CHECKPOINT)} {str(LOCAL_PLAIN_CHECKPOINT)}
    else:
        print("Plain checkpoint not found on Drive -- re-run notebook 03's decompression")
        print("cell here first (copy it in), or skip this comparison cell.")

del model
torch.cuda.empty_cache()
baseline_memory_bytes_2 = _gpu_memory_used_mb()

if LOCAL_PLAIN_CHECKPOINT.exists():
    model_plain = AutoModelForCausalLM.from_pretrained(
        str(LOCAL_PLAIN_CHECKPOINT), dtype=torch.float16, trust_remote_code=True, device_map='cuda',
    )
    plain_memory_bytes = _gpu_memory_used_mb()
    plain_delta_bytes = plain_memory_bytes - baseline_memory_bytes_2
    print(f"Delta (plain checkpoint's own footprint): {plain_delta_bytes / 1024**3:.3f} GiB")
    del model_plain
    torch.cuda.empty_cache()

## Log results

In [ ]:
REPO_DIR = Path('.').resolve()
RESULTS_DIR = REPO_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

_plain_delta = globals().get('plain_delta_bytes')

record = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'experiment': 'compressed_checkpoint_standalone_memory',
    'model': 'checkpoints/qwen3-1.7b-sgt-qat (compressed)',
    'compressed_checkpoint_delta_bytes': compressed_delta_bytes,
    'compressed_checkpoint_delta_gib': round(compressed_delta_bytes / 1024**3, 3),
    'plain_checkpoint_delta_bytes': _plain_delta,
    'plain_checkpoint_delta_gib': round(_plain_delta / 1024**3, 3) if _plain_delta is not None else None,
    'gpu': torch.cuda.get_device_name(0),
    'notes': (
        'Standalone weight-only load via transformers, NOT inside vLLM serving -- '
        'not directly comparable to notebook 02/03 GPU memory numbers (those '
        'include vLLM serving overhead / KV cache for a full request). See '
        'docs/findings.md for the caveat this is meant to partially address.'
    ),
}

ts = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%S')
fname = f"compressed_checkpoint_memory_{ts}.json"
(RESULTS_DIR / fname).write_text(json.dumps(record, indent=2))
print(f"Saved: results/{fname}")
print("\nTranscribe into docs/findings.md alongside the notebook 03 entry once this looks sane.")